# AeroQuant Lab — Visão Geral do Pipeline (Fases 1-5)

Notebook de demonstração reusando o código real e testado em `src/aeroquant/` (não é pseudocódigo — os mesmos módulos usados em `tests/` e `scripts/`).

**Pré-requisito**: rodar a partir da raiz do repositório com `PYTHONPATH=src`, ou descomentar a célula de `sys.path` abaixo.

In [ ]:
import sys
sys.path.insert(0, '../src')  # descomente/ajuste se não estiver usando PYTHONPATH=src

import matplotlib.pyplot as plt
import pandas as pd

## 1. Gerador Sintético (Fase 4, Nível 1)

Gera uma frota de unidades com vidas úteis variáveis, degradação via processo Gamma, ruído, deriva e falhas abruptas/intermitentes.

In [ ]:
from aeroquant.sensor_data.application.use_cases import GenerateSyntheticFleet
from aeroquant.sensor_data.domain.value_objects import DegradationParams
from aeroquant.sensor_data.infrastructure.cmapss_schema import build_cmapss_like_schema
from aeroquant.sensor_data.infrastructure.generators.stochastic_generator import StochasticSensorGenerator
from aeroquant.sensor_data.infrastructure.repositories.csv_repository import CSVSensorRepository

schema = build_cmapss_like_schema()
generator = StochasticSensorGenerator()
repo = CSVSensorRepository('/tmp/notebook_fleet.csv')

use_case = GenerateSyntheticFleet(generator, repo)
result = use_case.run(schema, DegradationParams(noise_std=0.015), n_units=20, lifetime_mean=180, lifetime_std=30, seed=123)
result

## 2. Qualidade de Dados + ETL (Fase 3)

In [ ]:
from aeroquant.sensor_data.etl.pipeline import clean, normalize, engineer_features, add_rul_labels, readings_to_dataframe, select_features
from aeroquant.sensor_data.infrastructure.quality.checks import DataQualityChecker

readings = repo.load()
df = readings_to_dataframe(readings, schema)

report = DataQualityChecker(schema).run(df)
print(f'Linhas: {report.n_rows} | Passou: {report.passed} | Issues: {len(report.issues)}')

In [ ]:
df = clean(df, schema)
df = normalize(df, schema)
df = engineer_features(df, schema, window=5)
df = add_rul_labels(df, max_rul_cap=125)
df.head()

## 3. Digital Twin (Fase 5)

Simula streaming cycle-by-cycle de UMA unidade e acompanha RUL previsto (com intervalo) vs. verdadeiro, além de detecção de anomalia.

In [ ]:
from aeroquant.digital_twin.application.use_cases import UpdateDigitalTwin
from aeroquant.digital_twin.infrastructure.estimators.linear_extrapolation_rul import LinearExtrapolationRULEstimator
from aeroquant.digital_twin.infrastructure.estimators.welford_fleet_baseline import OnlineFleetBaseline
from aeroquant.digital_twin.infrastructure.estimators.zscore_health_index import ZScoreHealthIndexEstimator
from aeroquant.digital_twin.infrastructure.estimators.threshold_calibration import calibrate_failure_threshold
from aeroquant.digital_twin.infrastructure.repositories.in_memory_repository import InMemoryDigitalTwinRepository
from aeroquant.sensor_data.domain.entities import FaultMode, Unit

hi_estimator = ZScoreHealthIndexEstimator(schema, coupling_threshold=0.2)
failure_threshold = calibrate_failure_threshold(schema, OnlineFleetBaseline(), hi_estimator)
print(f'Limiar de falha calibrado: {failure_threshold:.3f}')

In [ ]:
unit = Unit(unit_id='notebook-unit', fleet_id='f1', max_cycles=170, fault_mode=FaultMode.ABRUPT)
params = DegradationParams(seed=42, noise_std=0.012, abrupt_fault_rate=0.006, abrupt_fault_magnitude=0.5)
unit_readings = generator.generate_unit(unit, schema, params)

dt = UpdateDigitalTwin(
    baseline_tracker=OnlineFleetBaseline(),
    hi_estimator=hi_estimator,
    rul_estimator=LinearExtrapolationRULEstimator(min_points=8, window=40),
    repository=InMemoryDigitalTwinRepository(),
    healthy_window_cycles=20,
)

rows = []
for r in unit_readings:
    snap = dt.ingest(unit.unit_id, r.cycle, r.operating_condition, r.values, failure_threshold)
    rows.append({'cycle': r.cycle, 'true_rul': unit.max_cycles - r.cycle,
                 'pred_rul': snap.rul.point, 'lower': snap.rul.lower, 'upper': snap.rul.upper,
                 'anomaly': snap.is_anomaly})
dt_df = pd.DataFrame(rows)
dt_df.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(dt_df['cycle'], dt_df['true_rul'], label='RUL verdadeiro', color='black')
ax.plot(dt_df['cycle'], dt_df['pred_rul'], label='RUL previsto', color='tab:blue')
ax.fill_between(dt_df['cycle'], dt_df['lower'], dt_df['upper'], alpha=0.2, color='tab:blue')
for c in dt_df.loc[dt_df['anomaly'], 'cycle']:
    ax.axvline(c, color='tab:red', linestyle='--', alpha=0.5)
ax.set_ylim(0, 250); ax.set_xlabel('Ciclo'); ax.set_ylabel('RUL'); ax.legend()
plt.show()

## Próximos passos (não implementados neste notebook)

- Fase 6: substituir `LinearExtrapolationRULEstimator` por um modelo de ML (LSTM/ensemble) implementando o mesmo `RULEstimator` Protocol — comparar métricas contra este baseline.
- Fase 4, Nível 2: repetir este notebook usando `CMAPSSAdapter` sobre dados reais (`data/external/`) assim que os arquivos C-MAPSS forem baixados.

Ver `docs/roadmap.md` para o plano completo.